# 02b - Full-Feature Baselines (RF and LSTM-AE)

**Thesis:** Structured Subsystem-Aware Feature Representations for Predictive Maintenance in Metro Systems
**Author:** Emirhan Kurtulus, TU Wien

Reproduces the two MetroAT baselines of Steiner et al. (RAMS 2026) with Helm's code (`references/stefan/`) on the datasets of NB02a, using the **full feature space** (all sensors in one model). These are the full-feature side of the RQ2 comparison.
1. **Random Forest** (supervised): grid search with blocked CV, threshold tuned on the dev holdout, predictions on ALL test windows
2. **LSTM-AE** (unsupervised): trained on normal-only train sequences, reconstruction-error anomaly scores on ALL test sequences

**Outputs:** `outputs/supervised/` (`rf_grid_search.json`, `rf_model.joblib`, `test_predictions_rf.csv`), `outputs/unsupervised/` (`lstm_ae_model.pt`, `test_predictions_lstmae.csv`) and `outputs/anomaly_dataset_summary.json`.

> **Status (2026-10-27): not executed yet, planned for Phase 2.** The RF outputs currently in `outputs/supervised/` (`rf_model.joblib`, `test_predictions_rf.csv`) come from the earlier version of this code, which used fixed parameters without the grid search and refitted on the full undersampled set (test F1 = 0.120). NB03 and NB05 use those outputs until this notebook is run. Before running it, rebuild the LSTM-AE sequences in NB02a (see section 2); after running it, re-run NB03 and NB05.


In [ ]:
import sys, os, gc, itertools, json, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

ROOT = os.path.abspath('..')
OUT_DIR = os.path.join(ROOT, 'outputs')

# ── Steiner's best config (must match NB02a) ──
WINDOW_S  = 350      # 6 min feature aggregation window
HORIZON_S = 22326    # 6.2 h failure label shift

# ── Dataset description from NB02a's outputs ──
sup_cols = pq.read_schema(os.path.join(OUT_DIR, 'supervised', 'train.parquet')).names
N_ANALOG = sum(c.endswith('_min') for c in sup_cols)
N_BINARY = sum(c.endswith('_active_s') for c in sup_cols)
N_FEATURES = len([c for c in sup_cols if c not in {'timestamp', 'window_end', 'label'}])

with open(os.path.join(OUT_DIR, 'unsupervised', 'metadata.json')) as f:
    seq_meta = json.load(f)
UNSUP_CHANNELS = seq_meta['channels']
SEQ_WINDOW_S = seq_meta['seq_len_s']   # 1402 s = 23 min
SUBSAMPLE = seq_meta['subsample']      # every 4th timestep -> 350 steps

print(f'Supervised features: {N_FEATURES} ({N_ANALOG} analog sensors x 4, {N_BINARY} binary sensors x 2, 2 asset)')
print(f'LSTM-AE channels: {len(UNSUP_CHANNELS)} | sequence: {SEQ_WINDOW_S} s, subsample {SUBSAMPLE}')

# ── Cache flags: expensive steps reuse saved outputs unless explicitly re-run ──
RETRAIN_AE = False         # re-train the LSTM-AE (otherwise load the saved weights)
RERUN_GRID_SEARCH = False  # re-run the RF grid search (otherwise load outputs/supervised/rf_grid_search.json)

## 1. Random Forest Training

Replicates Helm's `train_rf_adapted.py` (`run_cv_grid`) exactly:
- Undersample the majority class to 10% positives
- Last 20% (temporal) of the undersampled set as holdout, the first 80% as dev set
- Grid search over 432 combinations with 5-fold blocked (expanding-window) CV on the dev set, selected by mean fold F1 at each fold's best threshold
- Refit the best combination on the dev set, tune the decision threshold on the holdout
- Predict ALL test windows with this dev-fitted model and threshold (Helm does not refit on the holdout)

The grid search takes about an hour, so its result is cached in `outputs/supervised/rf_grid_search.json` (flag `RERUN_GRID_SEARCH`).

In [ ]:
# ── RF Training: Helm's grid search with blocked CV ──
sup_train = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'train.parquet'))
sup_train['timestamp'] = pd.to_datetime(sup_train['timestamp'])
sup_train['window_end'] = pd.to_datetime(sup_train['window_end'])
sup_train = sup_train.sort_values('window_end').reset_index(drop=True)

LABEL_COL = 'label'
TIME_COL  = 'window_end'
ID_COLS   = {'timestamp', 'window_end'}
MIN_POS_RATIO = 0.10
N_FOLDS = 5

# Helm's PARAM_GRID (train_rf_adapted.py): 2 x 4 x 3 x 3 x 3 x 2 = 432 combinations
PARAM_GRID = {
    'n_estimators'     : [300, 600],
    'max_depth'        : [None, 2, 4, 16],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2', 0.5],
    'class_weight'     : [None, 'balanced'],
}

feature_cols = [c for c in sup_train.columns if c not in ID_COLS | {LABEL_COL}]
print(f'Feature columns: {len(feature_cols)}')


RNG = np.random.RandomState(42)  # local seed so this cell gives the same sample when re-run alone


def undersample(df, min_pos_ratio, rng):
    pos_idx = np.flatnonzero(df[LABEL_COL] == 1)
    neg_idx = np.flatnonzero(df[LABEL_COL] == 0)
    n_pos = len(pos_idx)
    if n_pos == 0 or n_pos / (n_pos + len(neg_idx)) >= min_pos_ratio:
        return df
    max_neg = int(np.floor(n_pos * (1.0 / min_pos_ratio - 1.0)))
    keep_neg = rng.choice(neg_idx, size=min(len(neg_idx), max_neg), replace=False)
    keep = np.sort(np.concatenate([pos_idx, keep_neg]))
    return df.iloc[keep]


def eval_best_threshold(y_true, y_proba):
    best_f1, best_thr, best_p, best_r = -1, 0.5, 0, 0
    for thr in np.linspace(0.01, 0.99, 99):
        yp = (y_proba >= thr).astype(int)
        f1 = f1_score(y_true, yp, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
            best_p = precision_score(y_true, yp, zero_division=0)
            best_r = recall_score(y_true, yp, zero_division=0)
    return best_thr, best_f1, best_p, best_r


def make_blocked_folds(times, n_folds):
    """Helm's make_blocked_folds: each block is validated by a model trained on all earlier windows."""
    times = pd.to_datetime(times)
    splits = []
    for val_idx in np.array_split(np.arange(len(times)), n_folds):
        if val_idx.size == 0:
            continue
        train_idx = np.where(times < times[val_idx[0]])[0]
        if train_idx.size > 0:
            splits.append((train_idx, val_idx))
    return splits


def make_model(params):
    pre = ColumnTransformer(
        transformers=[('pre', Pipeline([('impute', SimpleImputer(strategy='median', keep_empty_features=True))]), feature_cols)],
        verbose_feature_names_out=False
    )
    return Pipeline([('pre', pre), ('clf', RandomForestClassifier(random_state=42, n_jobs=-1, **params))])


pdf = undersample(sup_train, MIN_POS_RATIO, RNG)
print(f'After undersampling: {len(pdf)} windows (pos ratio: {(pdf[LABEL_COL] == 1).mean():.3f})')
holdout_size = int(0.2 * len(pdf))
pdf_dev = pdf.iloc[:-holdout_size].reset_index(drop=True)
pdf_holdout = pdf.iloc[-holdout_size:].reset_index(drop=True)
print(f'Dev: {len(pdf_dev)} | Holdout: {len(pdf_holdout)} (holdout positives: {int(pdf_holdout[LABEL_COL].sum())})')
X_dev, y_dev = pdf_dev[feature_cols], pdf_dev[LABEL_COL].astype(int)

GRID_PATH = os.path.join(OUT_DIR, 'supervised', 'rf_grid_search.json')
if RERUN_GRID_SEARCH or not os.path.exists(GRID_PATH):
    splits = make_blocked_folds(pdf_dev[TIME_COL].values, N_FOLDS)
    keys, values = zip(*PARAM_GRID.items())
    combos = [dict(zip(keys, v)) for v in itertools.product(*values)]
    print(f'Grid search: {len(combos)} combinations x {len(splits)} folds')
    grid_results = []
    for k, params in enumerate(combos, start=1):
        f1s = []
        for tr_idx, va_idx in splits:
            y_tr, y_va = y_dev.iloc[tr_idx], y_dev.iloc[va_idx]
            if y_tr.sum() == 0 or y_va.sum() == 0:
                f1s.append(0.0)
                continue
            m = make_model(params).fit(X_dev.iloc[tr_idx], y_tr)
            f1s.append(eval_best_threshold(y_va.values, m.predict_proba(X_dev.iloc[va_idx])[:, 1])[1])
        grid_results.append({'params': params, 'mean_f1': float(np.mean(f1s)), 'fold_f1': f1s})
        if k % 24 == 0:
            print(f'  {k}/{len(combos)} done, best mean F1 so far: {max(r["mean_f1"] for r in grid_results):.3f}', flush=True)
    best = max(grid_results, key=lambda r: r['mean_f1'])  # first maximum, like Helm's strict '>'
    with open(GRID_PATH, 'w') as f:
        json.dump({'best_params': best['params'], 'best_mean_cv_f1': best['mean_f1'],
                   'n_folds_used': len(splits), 'results': grid_results}, f, indent=2)
else:
    print(f'Loaded cached grid search result from {GRID_PATH}')

with open(GRID_PATH) as f:
    grid = json.load(f)
BEST_PARAMS = grid['best_params']
print(f"Best params: {BEST_PARAMS} (mean CV F1 = {grid['best_mean_cv_f1']:.3f})")

# Refit on the dev set and tune the threshold on the holdout (Helm's run_cv_grid)
final_model = make_model(BEST_PARAMS).fit(X_dev, y_dev)
proba_holdout = final_model.predict_proba(pdf_holdout[feature_cols])[:, 1]
best_thr, f1_h, p_h, r_h = eval_best_threshold(pdf_holdout[LABEL_COL].astype(int).values, proba_holdout)
print(f'Dev holdout: F1={f1_h:.3f} P={p_h:.3f} R={r_h:.3f} thr={best_thr:.3f}')

model_path = os.path.join(OUT_DIR, 'supervised', 'rf_model.joblib')
joblib.dump({'model': final_model, 'threshold': best_thr, 'params': BEST_PARAMS,
             'dev_metrics': {'F1': f1_h, 'P': p_h, 'R': r_h, 'thr': best_thr}},
            model_path)
print(f'Model saved to {model_path}')

del pdf, pdf_dev, pdf_holdout, X_dev, y_dev
gc.collect()


In [ ]:
# ── RF Predict on ALL test windows ──
saved = joblib.load(os.path.join(OUT_DIR, 'supervised', 'rf_model.joblib'))
final_model = saved['model']
best_thr = saved['threshold']

sup_test = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'test.parquet'))
sup_test['timestamp'] = pd.to_datetime(sup_test['timestamp'])
sup_test['window_end'] = pd.to_datetime(sup_test['window_end'])

feature_cols = [c for c in sup_test.columns if c not in {'timestamp', 'window_end', 'label'}]

X_test = sup_test[feature_cols]
y_test = sup_test['label'].astype(int)

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= best_thr).astype(int)

# Metrics
f1_t = f1_score(y_test, pred_test, zero_division=0)
p_t = precision_score(y_test, pred_test, zero_division=0)
r_t = recall_score(y_test, pred_test, zero_division=0)
cm = confusion_matrix(y_test, pred_test)
tn, fp, fn, tp = cm.ravel()
far = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'=== RF Test Results (threshold={best_thr:.3f}) ===')
print(f'  F1={f1_t:.3f}  P={p_t:.3f}  R={r_t:.3f}  FAR={far:.4f}')
print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'  True failures: {y_test.sum()} | Predicted: {pred_test.sum()}')

# Build output CSV
out_df = sup_test.copy()
out_df['true_failure_label'] = y_test.values
out_df['predicted_failure_label'] = pred_test
out_df['probability_of_failure'] = proba_test
out_df.drop(columns=['label'], inplace=True)

out_df.to_csv(os.path.join(OUT_DIR, 'supervised', 'test_predictions_rf.csv'), index=False)
print(f'\nSaved RF predictions: {len(out_df)} windows to outputs/supervised/test_predictions_rf.csv')

del X_test; gc.collect()

## 2. LSTM-AE Training

Simple LSTM autoencoder trained on normal-only sequences.
Anomaly score = mean reconstruction error per sequence.

**Phase 2 checklist (Helm alignment):** NB02a's sequences now follow Helm's code (3.1 h label at the sequence end, normal-only training set without the 3.1 h before failures, `dev_eval` hold-out). Rebuild them first (`REBUILD_SEQUENCES = True` in NB02a), then set `RETRAIN_AE = True` here. Helm chooses the alarm threshold on `dev_eval.npy` / `dev_eval_labels.npy`; add that step here so the LSTM-AE gets a test F1 comparable to Steiner's 0.08.


In [ ]:
# ── LSTM-AE Model Definition ──
torch.manual_seed(42)
np.random.seed(42)
HIDDEN_DIM = 64
N_EPOCHS = 10  # reduced from 20 for CPU feasibility
BATCH_SIZE = 128  # larger batches = fewer forward passes
LR = 1e-3
N_CHANNELS = len(UNSUP_CHANNELS)


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_channels, hidden_dim):
        super().__init__()
        self.encoder = nn.LSTM(n_channels, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, n_channels)
    
    def forward(self, x):
        _, (h, c) = self.encoder(x)
        seq_len = x.size(1)
        decoder_input = h.squeeze(0).unsqueeze(1).repeat(1, seq_len, 1)
        decoder_out, _ = self.decoder(decoder_input, (h, c))
        reconstruction = self.output_layer(decoder_out)
        return reconstruction


print(f'LSTM-AE config: hidden={HIDDEN_DIM}, epochs={N_EPOCHS}, batch={BATCH_SIZE}, channels={N_CHANNELS}')
print(f'Sequence length: {SEQ_WINDOW_S // SUBSAMPLE}')

In [ ]:
# ── Train LSTM-AE (or load the saved weights) ──
AE_PATH = os.path.join(OUT_DIR, 'unsupervised', 'lstm_ae_model.pt')
if RETRAIN_AE or not os.path.exists(AE_PATH):
    # ── Train LSTM-AE ──
    # Data from cell-10 is already subsampled to (N, 350, 95) — no further subsampling needed
    unsup_train = np.load(os.path.join(OUT_DIR, 'unsupervised', 'train.npy'))
    unsup_val   = np.load(os.path.join(OUT_DIR, 'unsupervised', 'val.npy'))

    print(f'Training data: {unsup_train.shape}')
    print(f'Validation data: {unsup_val.shape}')

    # Replace NaN with 0
    train_data = np.nan_to_num(unsup_train, 0.0)
    val_data   = np.nan_to_num(unsup_val, 0.0)

    # Create DataLoaders
    train_tensor = torch.FloatTensor(train_data)
    val_tensor   = torch.FloatTensor(val_data)

    train_loader = DataLoader(TensorDataset(train_tensor, train_tensor), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TensorDataset(val_tensor, val_tensor), batch_size=BATCH_SIZE)

    # Initialize model
    device = torch.device('cpu')
    model_ae = LSTMAutoencoder(N_CHANNELS, HIDDEN_DIM).to(device)
    optimizer = torch.optim.Adam(model_ae.parameters(), lr=LR)
    criterion = nn.MSELoss()

    # Training loop
    print('\nTraining LSTM-AE...')
    train_losses = []
    val_losses = []

    for epoch in range(N_EPOCHS):
        model_ae.train()
        epoch_loss = 0
        n_batches = 0
        for x_batch, _ in train_loader:
            x_batch = x_batch.to(device)
            recon = model_ae(x_batch)
            loss = criterion(recon, x_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        train_loss = epoch_loss / n_batches
        train_losses.append(train_loss)
    
        model_ae.eval()
        val_loss = 0
        n_val = 0
        with torch.no_grad():
            for x_batch, _ in val_loader:
                x_batch = x_batch.to(device)
                recon = model_ae(x_batch)
                val_loss += criterion(recon, x_batch).item()
                n_val += 1
        val_loss = val_loss / n_val
        val_losses.append(val_loss)
    
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:2d}/{N_EPOCHS}: train_loss={train_loss:.6f} val_loss={val_loss:.6f}')

    # Save model
    torch.save(model_ae.state_dict(), os.path.join(OUT_DIR, 'unsupervised', 'lstm_ae_model.pt'))
    print('\nLSTM-AE training complete. Model saved.')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_losses, label='Train loss')
    ax.plot(val_losses, label='Val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('LSTM-AE Training Curve')
    ax.legend()
    plt.tight_layout()
    plt.show()

    del unsup_train, unsup_val, train_data, val_data, train_tensor, val_tensor
    gc.collect()
else:
    print(f'Using saved LSTM-AE weights: {AE_PATH} (set RETRAIN_AE = True to re-train)')

In [ ]:
# ── Compute LSTM-AE anomaly scores on test sequences ──
# Data is already subsampled to (N, 350, 95) — no further subsampling needed
test_sequences = np.load(os.path.join(OUT_DIR, 'unsupervised', 'test.npy'))
test_seq_labels = np.load(os.path.join(OUT_DIR, 'unsupervised', 'test_labels.npy'))
test_seq_times_arr = np.load(os.path.join(OUT_DIR, 'unsupervised', 'test_times.npy'), allow_pickle=True)

test_clean_data = np.nan_to_num(test_sequences, 0.0)
test_tensor = torch.FloatTensor(test_clean_data)
test_loader = DataLoader(TensorDataset(test_tensor, test_tensor), batch_size=BATCH_SIZE)

# Load model
model_ae = LSTMAutoencoder(N_CHANNELS, HIDDEN_DIM)
model_ae.load_state_dict(torch.load(os.path.join(OUT_DIR, 'unsupervised', 'lstm_ae_model.pt'), weights_only=True))
model_ae.eval()

# Compute reconstruction errors
print('Computing reconstruction errors on test sequences...')
all_errors = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        recon = model_ae(x_batch)
        errors = ((x_batch - recon) ** 2).mean(dim=(1, 2)).numpy()
        all_errors.append(errors)

anomaly_scores = np.concatenate(all_errors)
print(f'Anomaly scores: {anomaly_scores.shape}')
print(f'  Normal  mean: {anomaly_scores[test_seq_labels == 0].mean():.6f}')
print(f'  Failure mean: {anomaly_scores[test_seq_labels == 1].mean():.6f}')

# Save LSTM-AE test results
lstmae_results = pd.DataFrame({
    'window_end': test_seq_times_arr,
    'anomaly_score_lstmae': anomaly_scores,
    'true_failure_label': test_seq_labels,
})
lstmae_results.to_csv(os.path.join(OUT_DIR, 'unsupervised', 'test_predictions_lstmae.csv'), index=False)
print(f'\nSaved LSTM-AE predictions: {len(lstmae_results)} sequences')

del test_sequences, test_clean_data, test_tensor; gc.collect()

## 3. Baseline Summary

RF and LSTM-AE test results side by side (kept separate: 6-min windows vs 23-min sequences).

In [ ]:
# ── Load both prediction sets ──
rf_preds = pd.read_csv(os.path.join(OUT_DIR, 'supervised', 'test_predictions_rf.csv'))
rf_preds['window_end'] = pd.to_datetime(rf_preds['window_end'])
rf_preds['timestamp'] = pd.to_datetime(rf_preds['timestamp'])

lstmae_preds = pd.read_csv(os.path.join(OUT_DIR, 'unsupervised', 'test_predictions_lstmae.csv'))
lstmae_preds['window_end'] = pd.to_datetime(lstmae_preds['window_end'])

print(f'RF predictions: {len(rf_preds)} windows (Wfeat={WINDOW_S}s)')
print(f'LSTM-AE predictions: {len(lstmae_preds)} sequences (Wseq={SEQ_WINDOW_S}s)')

# Note: RF and LSTM-AE have different temporal resolutions
# RF: 350s windows, LSTM-AE: 1402s windows
# We keep them separate but save a summary

print('\n=== RF Test Summary ===')
print(f'  Total windows: {len(rf_preds)}')
print(f'  True failures: {rf_preds["true_failure_label"].sum()}')
print(f'  Predicted failures: {rf_preds["predicted_failure_label"].sum()}')
print(f'  Date range: {rf_preds["timestamp"].min()} to {rf_preds["timestamp"].max()}')

print('\n=== LSTM-AE Test Summary ===')
print(f'  Total sequences: {len(lstmae_preds)}')
print(f'  True failures: {lstmae_preds["true_failure_label"].sum()}')
print(f'  Mean anomaly score (normal):  {lstmae_preds.loc[lstmae_preds["true_failure_label"]==0, "anomaly_score_lstmae"].mean():.6f}')
print(f'  Mean anomaly score (failure): {lstmae_preds.loc[lstmae_preds["true_failure_label"]==1, "anomaly_score_lstmae"].mean():.6f}')

In [ ]:
# ── Timeline visualization ──
saved = joblib.load(os.path.join(OUT_DIR, 'supervised', 'rf_model.joblib'))
rf_threshold = saved['threshold']

fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharex=True)

# Plot 1: RF probability of failure
ax = axes[0]
ax.scatter(rf_preds.loc[rf_preds['true_failure_label']==0, 'window_end'],
           rf_preds.loc[rf_preds['true_failure_label']==0, 'probability_of_failure'],
           s=1, alpha=0.3, c='blue', label='Normal')
ax.scatter(rf_preds.loc[rf_preds['true_failure_label']==1, 'window_end'],
           rf_preds.loc[rf_preds['true_failure_label']==1, 'probability_of_failure'],
           s=5, alpha=0.8, c='red', label='Failure')
ax.axhline(y=rf_threshold, color='green', linestyle='--', linewidth=1, label=f'Threshold={rf_threshold:.2f}')
ax.set_ylabel('P(failure)')
ax.set_title('RF Failure Probability — Full Test Period')
ax.legend(loc='upper right')

# Plot 2: LSTM-AE anomaly score
ax = axes[1]
ax.scatter(lstmae_preds.loc[lstmae_preds['true_failure_label']==0, 'window_end'],
           lstmae_preds.loc[lstmae_preds['true_failure_label']==0, 'anomaly_score_lstmae'],
           s=1, alpha=0.3, c='blue', label='Normal')
ax.scatter(lstmae_preds.loc[lstmae_preds['true_failure_label']==1, 'window_end'],
           lstmae_preds.loc[lstmae_preds['true_failure_label']==1, 'anomaly_score_lstmae'],
           s=5, alpha=0.8, c='red', label='Failure')
ax.set_ylabel('Reconstruction Error')
ax.set_title('LSTM-AE Anomaly Score — Full Test Period')
ax.set_xlabel('Time')
ax.legend(loc='upper right')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
os.makedirs(os.path.join(OUT_DIR, 'nb02_artifacts'), exist_ok=True)
plt.savefig(os.path.join(OUT_DIR, 'nb02_artifacts', 'anomaly_timeline.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Timeline plot saved to outputs/nb02_artifacts/anomaly_timeline.png')

In [ ]:
# ── Save summary ──
# Compute RF test metrics from predictions
y_true_rf = rf_preds['true_failure_label'].astype(int)
y_pred_rf = rf_preds['predicted_failure_label'].astype(int)
f1_t = f1_score(y_true_rf, y_pred_rf, zero_division=0)
p_t = precision_score(y_true_rf, y_pred_rf, zero_division=0)
r_t = recall_score(y_true_rf, y_pred_rf, zero_division=0)
cm = confusion_matrix(y_true_rf, y_pred_rf)
tn, fp, fn, tp = cm.ravel()
far = fp / (fp + tn) if (fp + tn) > 0 else 0

summary = {
    'supervised': {
        'window_s': WINDOW_S,
        'horizon_s': HORIZON_S,
        'n_analog': N_ANALOG,
        'n_binary': N_BINARY,
        'n_features': N_FEATURES,
        'test_windows': int(len(rf_preds)),
        'test_true_failures': int(rf_preds['true_failure_label'].sum()),
        'test_predicted_failures': int(rf_preds['predicted_failure_label'].sum()),
        'rf_threshold': float(rf_threshold),
        'rf_test_f1': float(f1_t),
        'rf_test_precision': float(p_t),
        'rf_test_recall': float(r_t),
        'rf_test_far': float(far),
        'rf_best_params': saved['params'],
        'rf_dev_holdout_metrics': saved['dev_metrics'],
        'label_definition': 'max failure flag over [window_end, window_end + horizon) (Helm create_shifted_labels); filter rule removes nothing',
        'threshold_tuning': 'undersample to 10% positives, last 20% as holdout; 432-combination grid search with 5-fold blocked CV on the dev set; dev-fitted model scores the test set (Helm train_rf_adapted.py)',
    },
    'unsupervised': {
        'seq_window_s': SEQ_WINDOW_S,
        'n_channels': N_CHANNELS,
        'subsample': SUBSAMPLE,
        'hidden_dim': HIDDEN_DIM,
        'n_epochs': N_EPOCHS,
        'test_sequences': int(len(lstmae_preds)),
        'test_failure_sequences': int(lstmae_preds['true_failure_label'].sum()),
    }
}

with open(os.path.join(OUT_DIR, 'anomaly_dataset_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved to outputs/anomaly_dataset_summary.json')
print('\n=== Done ===')
print('\nOutputs:')
print('  outputs/supervised/rf_grid_search.json     — RF grid search results')
print('  outputs/supervised/test_predictions_rf.csv — RF predictions on ALL test windows')
print('  outputs/supervised/rf_model.joblib         — trained RF model')
print('  outputs/unsupervised/test_predictions_lstmae.csv — LSTM-AE anomaly scores')
print('  outputs/unsupervised/lstm_ae_model.pt      — trained LSTM-AE model')
print('  outputs/nb02_artifacts/anomaly_timeline.png — timeline visualization')
print('  outputs/anomaly_dataset_summary.json       — summary statistics')

## Traceability: Pipeline Position

**NB01 → NB02a → NB02b → NB03 → NB04 → NB05**

### What this notebook provides:
- RF full-feature baseline: Helm's 432-combination grid search with 5-fold blocked CV on the dev set, threshold tuned on the dev holdout, test predictions with the dev-fitted model (results in `anomaly_dataset_summary.json`)
- LSTM-AE full-feature baseline: reconstruction scores only, no threshold or F1 in this notebook (F1 ≈ 0.08 is the value reported by Steiner/Helm). NB05 thresholds the scores at the 99th percentile of the validation split.

### Outputs:
- outputs/supervised/rf_model.joblib → NB03
- outputs/supervised/test_predictions_rf.csv → NB05
- outputs/unsupervised/lstm_ae_model.pt → NB03, NB05
- outputs/unsupervised/test_predictions_lstmae.csv → NB05